# DC1 against NDVI, by commune

Pairs the commune-monthly PC1 score (DC1) from
`create_commune_svd_vecs.ipynb` with the household-location-weighted commune
NDVI series, both released in `data/commune_monthly.csv`, and reports the
correlation between them for each of the six study communes.

`data/ndvi_timeseries/` holds the unweighted polygon means at their native
16-day resolution; those are provided as the underlying satellite extraction
and are not the series modelled here (see `data/README.md`).

Produces the per-commune panels and the 3x2 panel figure used in the
manuscript. All paths are relative to this notebook.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns; sns.set()

DATA = "../../data"

In [ ]:
# ---------------------------------------------------------------------
# Commune-monthly NDVI and DC1, communes 1-6
# ---------------------------------------------------------------------
commune_monthly = pd.read_csv(f"{DATA}/commune_monthly.csv")
commune_lookup = pd.read_csv(f"{DATA}/commune_lookup.csv")
commune_name = dict(zip(commune_lookup["commune"], commune_lookup["commune_name"]))

commune_monthly["Date"] = pd.to_datetime(commune_monthly["YearMonth"])
commune_monthly = commune_monthly.sort_values(["commune", "Date"])

COMMUNES = sorted(commune_monthly["commune"].unique())
print(commune_lookup)
print(commune_monthly.groupby("commune").size())


def commune_series(comm):
    """NDVI and DC1 for one commune, complete cases only."""
    return (commune_monthly[commune_monthly["commune"] == comm]
            .dropna(subset=["ndvi", "PC1_Score"])
            .copy())

In [ ]:
# ---------------------------------------------------------------------
# Pearson correlation, NDVI vs DC1
# ---------------------------------------------------------------------
print("Pearson correlation between NDVI and DC1:")
print("=" * 42)

correlations = {}
for comm in COMMUNES:
    df = commune_series(comm)
    correlations[comm] = df["ndvi"].corr(df["PC1_Score"])
    print(f"Commune {comm} ({commune_name[comm]}):  r = {correlations[comm]:.3f}   n = {len(df)}")

In [ ]:
# ---------------------------------------------------------------------
# Global plot style
# ---------------------------------------------------------------------
plt.rcParams.update({
    "font.size": 25,
    "axes.facecolor": "white",
    "figure.facecolor": "white",
})

year_locator = mdates.YearLocator()
year_fmt = mdates.DateFormatter("%Y")


def style_axes(ax_ndvi, ax_pc1):
    """Year ticks, rotated labels, full black box - shared by both figures."""
    ax_ndvi.xaxis.set_major_locator(year_locator)
    ax_ndvi.xaxis.set_major_formatter(year_fmt)
    ax_ndvi.set_xlabel("Year")
    ax_ndvi.grid(False)
    ax_pc1.grid(False)
    ax_ndvi.tick_params(axis="x", which="both", bottom=True, labelbottom=True,
                        direction="out", length=6, width=1.2)
    for label in ax_ndvi.get_xticklabels():
        label.set_rotation(44)
        label.set_horizontalalignment("right")
    for ax in (ax_ndvi, ax_pc1):
        for side in ("top", "bottom", "left", "right"):
            ax.spines[side].set_visible(True)
            ax.spines[side].set_linewidth(1.2)
            ax.spines[side].set_edgecolor("black")

In [ ]:
# ---------------------------------------------------------------------
# One figure per commune
# ---------------------------------------------------------------------
for comm in COMMUNES:
    df = commune_series(comm)

    fig, ax_ndvi = plt.subplots(figsize=(8, 4.5))
    ax_pc1 = ax_ndvi.twinx()

    ax_ndvi.plot(df["Date"], df["ndvi"], color="red", linewidth=2.2)
    ax_ndvi.set_ylabel("NDVI [Unitless Ratio]")

    ax_pc1.plot(df["Date"], df["PC1_Score"], color="blue", linewidth=2.2)
    ax_pc1.set_ylabel("DC1")

    style_axes(ax_ndvi, ax_pc1)
    fig.tight_layout()
    fig.savefig(f"commune_{comm}_ndvi_vs_dc1.pdf", format="pdf", bbox_inches="tight")
    plt.show()

print("Finished plotting.")

In [ ]:
# ---------------------------------------------------------------------
# 3x2 panel figure, communes 1-6 in manuscript order
# ---------------------------------------------------------------------
fig, axs = plt.subplots(3, 2, figsize=(14, 15), dpi=600)
axs = axs.flatten()

for i, comm in enumerate(COMMUNES):
    df = commune_series(comm)

    ax_ndvi = axs[i]
    ax_pc1 = ax_ndvi.twinx()

    ax_ndvi.plot(df["Date"], df["ndvi"], color="red", linewidth=2.2)
    ax_ndvi.set_ylabel("NDVI", color="black", fontsize=22)
    ax_ndvi.tick_params(axis="y", labelcolor="black", labelsize=18)

    ax_pc1.plot(df["Date"], df["PC1_Score"], color="blue", linewidth=2.2)
    ax_pc1.set_ylabel("DC1", color="black", fontsize=22)
    ax_pc1.tick_params(axis="y", labelcolor="black", labelsize=18)

    style_axes(ax_ndvi, ax_pc1)
    ax_ndvi.set_xlabel("Year", fontsize=22)
    ax_ndvi.tick_params(axis="x", labelsize=18)

    # Panel label (a)-(f)
    ax_ndvi.text(0.03, 0.9, f"({chr(97+i)})", transform=ax_ndvi.transAxes,
                 fontsize=22, fontweight="bold",
                 bbox=dict(facecolor="white", edgecolor="none", alpha=0.7))

plt.subplots_adjust(hspace=0.35, wspace=0.55)
fig.savefig("ndvi_vs_dc1_3x2_panel.pdf", format="pdf", bbox_inches="tight")
plt.close(fig)

print("Saved 3x2 panel figure: ndvi_vs_dc1_3x2_panel.pdf")